In [2]:
!pip install viser



import cv2

import os

import numpy as np

from google.colab.patches import cv2_imshow

   ---------------------------------------- 0.0/29.0 MB ? eta -:--:--
    --------------------------------------- 0.5/29.0 MB 4.4 MB/s eta 0:00:07
   - -------------------------------------- 1.3/29.0 MB 3.5 MB/s eta 0:00:08
   -- ------------------------------------- 2.1/29.0 MB 3.9 MB/s eta 0:00:07
   ---- ----------------------------------- 3.1/29.0 MB 3.9 MB/s eta 0:00:07
   ----- ---------------------------------- 3.9/29.0 MB 4.0 MB/s eta 0:00:07
   ----- ---------------------------------- 4.2/29.0 MB 3.7 MB/s eta 0:00:07
   ------ --------------------------------- 4.5/29.0 MB 3.2 MB/s eta 0:00:08
   ------ --------------------------------- 4.5/29.0 MB 3.2 MB/s eta 0:00:08
   ------ --------------------------------- 4.7/29.0 MB 2.7 MB/s eta 0:00:10
   ------ --------------------------------- 4.7/29.0 MB 2.7 MB/s eta 0:00:10
   ------ --------------------------------- 5.0/29.0 MB 2.3 MB/s eta 0:00:11
   ------- -------------------------------- 5.2/29.0 MB 2.3 MB/s eta 0:00:11
   ---

ModuleNotFoundError: No module named 'cv2'

In [ ]:
# marker is 60mm
MARKER_WIDTH = 0.06

# coords for a single marker, origin at top-left
SINGLE_MARKER_COORDS = np.array([
    [0.0,          0.0,           0.0],
    [MARKER_WIDTH, 0.0,           0.0],
    [MARKER_WIDTH, MARKER_WIDTH,  0.0],
    [0.0,          MARKER_WIDTH,  0.0]
], dtype=np.float32)

# 3d position of the top-left corner of each of our 6 tags
GRID_ORIGINS_3D = np.array([
    [0.00, 0.00000, 0.0],
    [0.09, 0.00000, 0.0],
    [0.00, 0.07567, 0.0],
    [0.09, 0.07567, 0.0],
    [0.00, 0.15134, 0.0],
    [0.09, 0.15134, 0.0],
], dtype=np.float32)

# gives (6, 4, 3) array of all 3d world points for all 6 tags
ALL_MARKER_WORLD_POINTS = GRID_ORIGINS_3D[:, None, :] + SINGLE_MARKER_COORDS[None, :, :]

NameError: name 'np' is not defined

In [6]:
def create_aruco_detector():
    """sets up the 4x4 marker detector"""
    marker_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    detector_config = cv2.aruco.DetectorParameters()
    return cv2.aruco.ArucoDetector(marker_dict, detector_config)

In [3]:
def find_and_show_markers(img, marker_detector):
    """
    detects markers, shows a debug view, and returns pixel coords
    """
    detected_corners, detected_ids, _ = marker_detector.detectMarkers(img)

    # maps tag_id -> pixel coords
    marker_pixel_map = [None] * 6

    if detected_ids is not None:
        debug_img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

        # loop all detected
        for i, tag_id in enumerate(detected_ids.flatten()):
            if 0 <= tag_id < 6:
                # save the 2d pixel coords
                px_coords = detected_corners[i].reshape(4, 2).astype(np.float32)
                marker_pixel_map[tag_id] = px_coords

                # draw stuff for debug view
                int_pts = px_coords.astype(int)
                cv2.polylines(debug_img, [int_pts], isClosed=True, color=(0, 255, 0), thickness=5)
                x, y = int_pts[0]
                cv2.putText(debug_img, str(tag_id), (x, y - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2, cv2.LINE_AA)

        cv2_imshow(cv2.resize(debug_img, (1280, 700)))
        cv2.waitKey(1)

    return marker_pixel_map

In [ ]:
def gather_calibration_points(image_folder):
    """
    loads all images, finds markers, and builds 2d/3d point lists
    """
    all_world_pts = []
    all_pixel_pts = []
    image_size = None

    # setup aruco detector once
    marker_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    detector_config = cv2.aruco.DetectorParameters()
    detector = cv2.aruco.ArucoDetector(marker_dict, detector_config)

    print(f"progress: {0}")
    try:
        image_files = [f for f in os.listdir(image_folder) if f.endswith(".jpg")]
        total_images = len(image_files)
    except FileNotFoundError:
        print(f"error: folder not found: {image_folder}")
        return [], [], None

    for i, fname in enumerate(image_files):
        img_path = os.path.join(image_folder, fname)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            print(f"warning: could not read {fname}")
            continue
            
        if image_size is None:
            h, w = img.shape[:2]
            image_size = (w, h)

        # find markers in this one image
        found_pixels = find_and_show_markers(img, detector)
        print(f"progress: {(i + 1) / total_images * 0.8 + 0.05}") # map 0-1 to 0.05-0.85

        current_image_pixels = []
        current_image_world = []

        # check which of the 6 tags were found
        for tag_id, px in enumerate(found_pixels):
            if px is not None:
                current_image_pixels.append(px)
                current_image_world.append(ALL_MARKER_WORLD_POINTS[tag_id])

        # if we found at least one tag, add its points
        if current_image_pixels:
            # format for opencv
            cv_pix_pts = np.concatenate(current_image_pixels, axis=0).reshape(-1, 1, 2).astype(np.float32)
            cv_world_pts = np.concatenate(current_image_world, axis=0).reshape(-1, 1, 3).astype(np.float32)

            all_pixel_pts.append(cv_pix_pts)
            all_world_pts.append(cv_world_pts)

    return all_world_pts, all_pixel_pts, image_size

In [5]:
IMAGE_SOURCE_DIR = "calibration_imgs"
OUTPUT_FILE = "output.txt"

# phase 1: find all points
obj_points, img_points, img_dims = gather_calibration_points(IMAGE_SOURCE_DIR)

if not obj_points:
    print("no calibration points found. stopping.")

print(f"progress: {0.90}")

# phase 2: run calibration
reprojection_error, cam_matrix, dist_coeffs, rot_vecs, trans_vecs = cv2.calibrateCamera(
    obj_points, img_points, img_dims, None, None
)

print(f"progress: {0.95}")
print(f"image size: {img_dims[0]} {img_dims[1]}")

# phase 3: save results
with open(OUTPUT_FILE, "w") as f:
    f.write(f"Error= {reprojection_error}\n")
    f.write(f"K= {cam_matrix}\n")
    f.write(f"distortion= {dist_coeffs}\n")
    
print(f"calibration done. results saved to {OUTPUT_FILE}")

NameError: name 'cv2' is not defined